# Deep-learning avalanche detection vs. classical image processing

Trains a U-Net (ResNet-34 encoder, ImageNet-pretrained) to outline avalanches in a photo
and compares it with the classical pipeline of this repository on real webcam image pairs.

**Data**
- [UIBK Avalanche Dataset](https://researchdata.uibk.ac.at/records/h07f4-qzd17)
  (Fox et al. 2023, Apache 2.0): 4,090 field photos, avalanches outlined as polygons.
  Used for pre-training; its own test split is used to choose the decision threshold.
- [SLF avalanche webcam dataset](https://doi.org/10.16904/envidat.526)
  (Hafner 2024, CC BY-SA 4.0): fixed Davos webcams, expert masks. Used for fine-tuning
  and for the final comparison.

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
It takes about 60 minutes (two downloads of ~3 GB each, then training). Keep the tab open.
At the end two files are downloaded: `dl_results.zip` (evaluation) and `web_app.zip`
(the final model for the web interface); allow multiple downloads if the browser asks.

**Honest evaluation:** the SLF webcams are split into 3 folds by camera view
(*leave-camera-view-out*): every webcam image is evaluated by a model that never saw
that view. The threshold is chosen on UIBK data, never on the SLF test images.

In [ ]:
# 1) Setup
import subprocess, time, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))
!pip -q install segmentation-models-pytorch
![ -d /content/snow-detection ] || git clone -q https://github.com/bircanbektas/snow-detection.git /content/snow-detection
%cd /content/snow-detection

In [ ]:
# 2) Download both datasets (~3 GB each) and extract only images and annotations
import re, zipfile
from pathlib import Path

SLF_URL = ("https://www.envidat.ch/dataset/85ae804d-3282-4935-82b2-9e43d024c266/resource/"
           "e0d0a419-86e9-4fde-9254-410851363b80/download/ias_data.zip")
UIBK_URL = "https://researchdata.uibk.ac.at/records/h07f4-qzd17/files/uibk_avalanches.zip?download=1"
SPLITS = ("Train", "Vali", "Test", "WebNew", "GroundPic", "UserPic")
jobs = [  # url, zip, folder, which members to extract
    (SLF_URL, "/content/ias_data.zip", "/content/slf_raw",
     re.compile(r"(^|/)(%s)/(images|masks)/[^/]+$" % "|".join(SPLITS))),
    (UIBK_URL, "/content/uibk.zip", "/content/uibk_raw",
     re.compile(r"(^|/)(images/(glide|loose|slab|none)/[^/]+\.jpe?g|annotations/(glide|loose|slab)/[^/]+\.json"
                r"|train\.txt|test\.txt)$", re.I)),
]
for url, zip_path, folder, pattern in jobs:
    if Path(folder).exists():
        continue
    start = time.time()
    print("Downloading", zip_path, "...")
    subprocess.run(["wget", "-q", "-O", zip_path, url], check=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(folder, [m for m in z.namelist()
                              if pattern.search(m) and "__MACOSX" not in m])
    Path(zip_path).unlink()                       # free the disk space
    print(f"  done in {(time.time() - start) / 60:.0f} min")

ROOT = next(Path("/content/slf_raw").rglob("Train")).parent
UIBK = next(Path("/content/uibk_raw").rglob("annotations")).parent
for s in SPLITS:
    print("SLF", s, len(list((ROOT / s / "images").glob("*"))), "files")
for c in ("glide", "loose", "slab", "none"):
    print("UIBK", c, len(list((UIBK / "images" / c).glob("*"))), "images")

In [ ]:
# 3) Prepare all photos at width 1200 px with one avalanche mask each
import json, cv2, numpy as np
from collections import defaultdict
from multiprocessing import Pool

WIDTH = 1200

# SLF webcams: merge the per-avalanche masks, group the images by camera view (repo script)
!python evaluation/prepare_slf.py {ROOT} /content/prepared

# SLF hand-held photos (training only)
EXTRA = Path("/content/extra")
EXTRA.mkdir(exist_ok=True)
for split in ("GroundPic", "UserPic"):
    groups = defaultdict(lambda: {"image": None, "masks": []})
    masks = {m.stem: m for m in (ROOT / split / "masks").glob("*")}
    for img in (ROOT / split / "images").glob("*"):
        key = split + "_" + re.sub(r"_\d+$", "", img.stem)
        groups[key]["image"] = img
        if img.stem in masks:
            groups[key]["masks"].append(masks[img.stem])
    used = 0
    for key, g in groups.items():
        img = cv2.imread(str(g["image"]))
        if img is None or not g["masks"]:
            continue
        scale = WIDTH / img.shape[1]
        small = cv2.resize(img, (WIDTH, int(img.shape[0] * scale)), interpolation=cv2.INTER_AREA)
        union = np.zeros(small.shape[:2], np.uint8)
        for m in g["masks"]:
            mk = cv2.imread(str(m), cv2.IMREAD_GRAYSCALE)
            if mk is not None:
                union[cv2.resize(mk, small.shape[1::-1], interpolation=cv2.INTER_NEAREST) > 0] = 255
        cv2.imwrite(str(EXTRA / f"{key}.jpg"), small, [cv2.IMWRITE_JPEG_QUALITY, 92])
        cv2.imwrite(str(EXTRA / f"{key}_mask.png"), union)
        used += 1
    print(f"{split}: {used} photos with masks")

# UIBK: rasterize the polygons (VoTT JSON); "none" images get an empty mask
UPREP = Path("/content/uibk_prep")
UPREP.mkdir(exist_ok=True)
annotations = {f"{p.parent.name}/{p.stem}": p for p in (UIBK / "annotations").glob("*/*.json")}


def prepare_uibk(path):
    key = f"{path.parent.name}/{path.stem}"
    name = key.replace("/", "__")
    if (UPREP / f"{name}_mask.png").exists():
        return name, "ok"
    img = cv2.imread(str(path), cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
    if img is None:
        return name, "unreadable"
    mask = np.zeros(img.shape[:2], np.uint8)
    status = "ok"
    if key in annotations:
        ann = json.load(open(annotations[key], encoding="utf-8"))
        aw, ah = ann["asset"]["size"]["width"], ann["asset"]["size"]["height"]
        if (aw, ah) != (img.shape[1], img.shape[0]):
            rotated = cv2.imread(str(path))      # annotated with EXIF rotation applied?
            if (aw, ah) == (rotated.shape[1], rotated.shape[0]):
                img, mask = rotated, np.zeros(rotated.shape[:2], np.uint8)
            else:
                status = "rescaled"
        sx, sy = img.shape[1] / aw, img.shape[0] / ah
        for region in ann["regions"]:
            pts = np.array([[p["x"] * sx, p["y"] * sy] for p in region["points"]], np.int32)
            cv2.fillPoly(mask, [pts], 255)
    elif path.parent.name != "none":
        return name, "no annotation"
    scale = WIDTH / img.shape[1]
    size = (WIDTH, int(img.shape[0] * scale))
    cv2.imwrite(str(UPREP / f"{name}.jpg"), cv2.resize(img, size, interpolation=cv2.INTER_AREA),
                [cv2.IMWRITE_JPEG_QUALITY, 92])
    cv2.imwrite(str(UPREP / f"{name}_mask.png"), cv2.resize(mask, size, interpolation=cv2.INTER_NEAREST))
    return name, status


start = time.time()
with Pool(4) as pool:
    results = pool.map(prepare_uibk, sorted((UIBK / "images").glob("*/*")), chunksize=16)
print(f"UIBK prepared in {(time.time() - start) / 60:.0f} min:",
      {s: sum(r[1] == s for r in results) for s in set(r[1] for r in results)})


def read_list(name):
    lines = open(UIBK / name, encoding="utf-8").read().splitlines()
    keys = [l.strip().strip("/").rsplit(".", 1)[0].replace("/", "__") for l in lines if l.strip()]
    return [k for k in keys if (UPREP / f"{k}_mask.png").exists()]


uibk_train, uibk_test = read_list("train.txt"), read_list("test.txt")
print(len(uibk_train), "UIBK training photos,", len(uibk_test), "UIBK test photos")

In [ ]:
# 4) Data, model and training code
import random
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

PREP = Path("/content/prepared")
idx = json.load(open(PREP / "index.json"))
webcam = [it["key"] for it in idx["items"]]
extra = sorted(p.stem for p in EXTRA.glob("*.jpg"))
files = {k: PREP / k for k in webcam}
files.update({k: EXTRA / k for k in extra})
files.update({k: UPREP / k for k in uibk_train + uibk_test})


def load(key):
    img = cv2.imread(str(files[key]) + ".jpg")
    mask = (cv2.imread(str(files[key]) + "_mask.png", 0) > 0).astype(np.float32)
    return img, mask


data = {k: load(k) for k in webcam}          # webcam images stay in memory for the evaluation
print(len(webcam), "webcam images,", len(extra), "SLF hand-held photos,",
      len(uibk_train), "UIBK training photos")

MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
CROP = 512


def to_tensor(img):
    rgb = (img[..., ::-1].astype(np.float32) / 255 - MEAN) / STD
    return torch.from_numpy(rgb.transpose(2, 0, 1).copy())


def random_crop(img, mask):
    s = random.uniform(0.6, 1.2)
    img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    mask = cv2.resize(mask, img.shape[1::-1], interpolation=cv2.INTER_NEAREST)
    ph, pw = max(CROP - img.shape[0], 0), max(CROP - img.shape[1], 0)
    if ph or pw:
        img = cv2.copyMakeBorder(img, 0, ph, 0, pw, cv2.BORDER_REFLECT)
        mask = cv2.copyMakeBorder(mask, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=0)
    h, w = mask.shape
    ys, xs = np.nonzero(mask)
    if len(ys) and random.random() < 0.6:          # avalanches are small: often crop around one
        i = random.randrange(len(ys))
        y0 = ys[i] - CROP // 2 + random.randint(-CROP // 4, CROP // 4)
        x0 = xs[i] - CROP // 2 + random.randint(-CROP // 4, CROP // 4)
    else:
        y0, x0 = random.randint(0, h - CROP), random.randint(0, w - CROP)
    y0, x0 = int(np.clip(y0, 0, h - CROP)), int(np.clip(x0, 0, w - CROP))
    img, mask = img[y0:y0 + CROP, x0:x0 + CROP], mask[y0:y0 + CROP, x0:x0 + CROP]
    if random.random() < 0.5:
        img, mask = img[:, ::-1], mask[:, ::-1]
    # lighting changes: contrast, brightness, gamma
    img = img.astype(np.float32) * random.uniform(0.7, 1.3) + random.uniform(-30, 30)
    img = 255 * (np.clip(img, 0, 255) / 255) ** random.uniform(0.7, 1.4)
    return img.astype(np.uint8), mask


class Crops(Dataset):
    def __init__(self, keys):
        self.keys = keys

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, i):
        img, mask = random_crop(*load(self.keys[i]))
        return to_tensor(img), torch.from_numpy(mask.copy())[None]


def loss_fn(logits, y):
    p = torch.sigmoid(logits)
    dice = 1 - (2 * (p * y).sum() + 1) / (p.sum() + y.sum() + 1)
    return F.binary_cross_entropy_with_logits(logits, y) + dice


def train(keys, epochs, lr=3e-4, init=None, seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = smp.Unet("resnet34", encoder_weights="imagenet", classes=1).cuda()
    if init is not None:
        model.load_state_dict(init)
    dl = DataLoader(Crops(keys), batch_size=8, shuffle=True, num_workers=2, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(dl))
    scaler = torch.amp.GradScaler()
    start = time.time()
    for ep in range(epochs):
        model.train()
        total = 0
        for x, y in dl:
            x, y = x.cuda(), y.cuda()
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(x)
            loss = loss_fn(logits.float(), y)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            total += loss.item()
        print(f"  epoch {ep + 1}/{epochs}  loss {total / len(dl):.3f}  {time.time() - start:.0f}s")
    return model.eval()


@torch.no_grad()
def predict(model, img):
    """Avalanche probability per pixel (average of the image and its mirror)."""
    h, w = img.shape[:2]
    padded = cv2.copyMakeBorder(img, 0, (-h) % 32, 0, (-w) % 32, cv2.BORDER_REFLECT)
    x = to_tensor(padded)[None].cuda()
    with torch.autocast("cuda", dtype=torch.float16):
        p = torch.sigmoid(model(x)) + torch.sigmoid(model(x.flip(-1))).flip(-1)
    return (p[0, 0, :h, :w] / 2).float().cpu().numpy()


def segmentation_scores(pairs, thr):
    """Pooled pixel scores over (probability map, ground-truth mask) pairs."""
    tp = fp = fn = 0
    for p, gt in pairs:
        pred, gt = p > thr, gt > 0
        tp, fp, fn = tp + np.sum(pred & gt), fp + np.sum(pred & ~gt), fn + np.sum(~pred & gt)
    return {"iou": tp / max(tp + fp + fn, 1), "f1": 2 * tp / max(2 * tp + fp + fn, 1),
            "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1)}

In [ ]:
# 5) Pre-train on UIBK + SLF hand-held photos (no SLF webcam image is used here)
base = train(uibk_train + extra, epochs=10)
base_state = {k: v.clone() for k, v in base.state_dict().items()}

# Choose the decision threshold on the UIBK test split (never on SLF webcams)
uibk_eval = [(predict(base, load(k)[0]), load(k)[1]) for k in uibk_test]
by_thr = {t: segmentation_scores(uibk_eval, t) for t in np.arange(0.1, 0.75, 0.05).round(2)}
THR = max(by_thr, key=lambda t: by_thr[t]["f1"])
uibk_scores = by_thr[THR]
print(f"threshold {THR}  UIBK test:", {k: round(v, 3) for k, v in uibk_scores.items()})
del uibk_eval

In [ ]:
# 6) Fine-tune on SLF webcams, leave-camera-view-out (3 folds)
view_of = {k: vi for vi, v in enumerate(idx["views"]) for k in v}
groups = defaultdict(list)
for k in webcam:
    groups[view_of.get(k, "single_" + k)].append(k)
folds = [[], [], []]
for g in sorted(groups.values(), key=len, reverse=True):
    min(folds, key=len).extend(g)

prob = {"pretrained": {k: predict(base, data[k][0]) for k in webcam}, "fine-tuned": {}}
del base
rng = random.Random(0)
for fi, test_keys in enumerate(folds):
    cams = [k for k in webcam if k not in test_keys]
    # webcams repeated so they are not drowned out; some UIBK photos so nothing is forgotten
    keys = cams * 10 + extra + rng.sample(uibk_train, 600)
    print(f"Fold {fi + 1}/3: fine-tuning on {len(cams)} webcam images, testing on {len(test_keys)}")
    model = train(keys, epochs=4, lr=1e-4, init=base_state, seed=fi)
    for k in test_keys:
        prob["fine-tuned"][k] = predict(model, data[k][0])
    del model
    torch.cuda.empty_cache()

seg = {"threshold": float(THR), "uibk_test": uibk_scores}
for name, p in prob.items():
    seg[name] = segmentation_scores([(p[k], data[k][1]) for k in webcam], THR)
    print(f"SLF webcams, {name:>10}:", {k: round(v, 3) for k, v in seg[name].items()})

In [ ]:
# 7) Same pair evaluation as evaluation/evaluate_slf.py: classical pipeline vs. models
import contextlib, io
from datetime import datetime
from avalanche_monitor import analyze

NEW_MIN_PCT = 0.2        # new avalanche area that makes a pair "positive" and raises a model alarm
orb, bf = cv2.ORB_create(5000), cv2.BFMatcher(cv2.NORM_HAMMING)
OUT = Path("/content/dl_results")
OUT.mkdir(exist_ok=True)


def homography(ref, mov):
    k1, d1 = orb.detectAndCompute(cv2.cvtColor(ref, cv2.COLOR_BGR2GRAY), None)
    k2, d2 = orb.detectAndCompute(cv2.cvtColor(mov, cv2.COLOR_BGR2GRAY), None)
    if d1 is None or d2 is None:
        return None, 0
    good = [p[0] for p in bf.knnMatch(d2, d1, k=2)
            if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
    if len(good) < 15:
        return None, 0
    src = np.float32([k2[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([k1[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, inl = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    return H, (int(inl.sum()) if H is not None else 0)


def new_regions(later, earlier):
    """Pixels marked in the later mask that were not marked (nearby) in the earlier one."""
    old = cv2.dilate(earlier, np.ones((15, 15), np.uint8))
    new = ((later > 0) & (old == 0)).astype(np.uint8) * 255
    return cv2.morphologyEx(new, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))


def scores(pred, gt):
    hit = np.count_nonzero((pred > 0) & (gt > 0))
    near = cv2.dilate(gt, np.ones((31, 31), np.uint8))
    recall = hit / max(np.count_nonzero(gt), 1) * 100
    precision = np.count_nonzero((pred > 0) & (near > 0)) / max(np.count_nonzero(pred), 1) * 100
    return recall, precision


binary = lambda m: (m > 0).astype(np.uint8) * 255
times = {it["key"]: it["time"] for it in idx["items"]}
rows = []
for view in idx["views"]:
    for a, b in zip(view, view[1:]):
        img_a, img_b = data[a][0], data[b][0]
        H, inliers = homography(img_a, img_b)
        if H is None or inliers < 60:
            continue
        h, w = img_a.shape[:2]
        gap_h = (datetime.strptime(times[b], "%Y%m%d%H%M")
                 - datetime.strptime(times[a], "%Y%m%d%H%M")).total_seconds() / 3600
        warp = lambda m: cv2.warpPerspective(m, H, (w, h), flags=cv2.INTER_NEAREST)
        gt = new_regions(warp(binary(data[b][1])), binary(data[a][1]))
        row = {"pair": f"{a} -> {b}", "gap_h": round(gap_h, 1),
               "positive": bool(np.count_nonzero(gt) / gt.size * 100 >= NEW_MIN_PCT)}

        with contextlib.redirect_stdout(io.StringIO()):
            r = analyze(img_a, img_b, roi=None, roi_file=None)
        masks = {"classical": cv2.bitwise_or(r["change_mask"], r["motion_mask"])}
        row["classical"] = {"alarm": r["risk_level"] != "LOW", "scores": scores(masks["classical"], gt)}
        for name, p in prob.items():
            masks[name] = new_regions(warp(binary(p[b] > THR)), binary(p[a] > THR))
            pct = np.count_nonzero(masks[name]) / gt.size * 100
            row[name] = {"alarm": bool(pct >= NEW_MIN_PCT), "new_pct": round(pct, 2),
                         "scores": scores(masks[name], gt)}
        rows.append(row)

        # picture: later image, green = real new avalanches, red = fine-tuned model, blue = classical
        vis = cv2.warpPerspective(img_b, H, (w, h))
        for m, color in ((masks["classical"], (255, 120, 0)), (masks["fine-tuned"], (0, 0, 255)),
                         (gt, (0, 255, 0))):
            cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(vis, cnts, -1, color, 3)
        cv2.imwrite(str(OUT / f"{a}__{b}.jpg"), cv2.resize(vis, None, fx=0.5, fy=0.5))


def summary(sel, label):
    pos = [r for r in sel if r["positive"]]
    neg = [r for r in sel if not r["positive"]]
    print(f"\n[{label}] {len(pos)} pairs with a new avalanche, {len(neg)} without")
    print(f"  {'method':<14}{'detected':>10}{'right place':>13}{'false alarms':>15}{'pixels found':>15}")
    for name in ("classical", "pretrained", "fine-tuned"):
        hits = [r for r in pos if r[name]["alarm"]]
        found = np.mean([r[name]["scores"][0] for r in pos]) if pos else float("nan")
        print(f"  {name:<14}{len(hits):>6} / {len(pos):<3}{sum(r[name]['scores'][0] > 0 for r in hits):>10}"
              f"{sum(r[name]['alarm'] for r in neg):>9} / {len(neg):<5}{found:>12.1f} %")


summary(rows, "all pairs")
summary([r for r in rows if r["gap_h"] <= 6], "pairs <= 6 hours apart")

In [ ]:
# 8) Download the results (summary + pictures) to put them into the repository
json.dump({"segmentation": seg, "pairs": rows, "folds": folds}, open(OUT / "results.json", "w"),
          indent=1, default=float)
!cd /content && zip -qr dl_results.zip dl_results
from google.colab import files as colab_files
colab_files.download("/content/dl_results.zip")

In [ ]:
# 9) Final model for the web app: fine-tune on all SLF webcams, export to ONNX (runs with OpenCV)
#    Input is a fixed 1216 x 928 canvas; the app resizes and pads the photo to it.
keys = webcam * 10 + extra + random.Random(1).sample(uibk_train, 600)
final = train(keys, epochs=4, lr=1e-4, init=base_state, seed=3).float().cpu()
APP = Path("/content/web_app")
APP.mkdir(exist_ok=True)
onnx_path = APP / "avalanche_unet.onnx"
dummy = torch.zeros(1, 3, 928, 1216)
try:
    torch.onnx.export(final, dummy, str(onnx_path), input_names=["image"], output_names=["logits"],
                      opset_version=17, dynamo=False)
except Exception as e:                          # newer PyTorch: only the dynamo exporter is left
    print("Legacy ONNX exporter failed:", e)
    subprocess.run(["pip", "-q", "install", "onnxscript"], check=True)
    torch.onnx.export(final, (dummy,), str(onnx_path), input_names=["image"],
                      output_names=["logits"], dynamo=True)

# check: OpenCV must give the same output as PyTorch
img = cv2.resize(data[webcam[0]][0], (1216, 928))
x = to_tensor(img)[None]
net = cv2.dnn.readNetFromONNX(str(onnx_path))
net.setInput(x.numpy())
with torch.no_grad():
    diff = np.abs(net.forward() - final(x).numpy()).max()
print(f"ONNX model {onnx_path.stat().st_size / 1e6:.0f} MB, max difference OpenCV vs PyTorch: {diff:.4f}")

# a real sample pair (2 hours apart, new avalanche) for the web app
for k in ("20210319093501_bb2", "20210319113501_bb2"):
    if k in data:
        cv2.imwrite(str(APP / f"slf_{k}.jpg"), data[k][0], [cv2.IMWRITE_JPEG_QUALITY, 90])
!cd /content && zip -qr web_app.zip web_app
colab_files.download("/content/web_app.zip")